# 009: Model Compression and Quantization — Weight pruning and integer quantization math

## 1. WEIGHT PRUNING
- Prunes redundant weights (e.g. settings weights below a threshold $\tau$ to zero):
  $$w = 0 \text{ if } |w| < \tau$$
- This yields sparse matrices, reducing memory usage and inference storage.

## 2. QUANTIZATION (FP32 → INT8)
- Maps 32-bit floating-point weights to 8-bit integers:
  $$q = \text{round}\left(\frac{r}{S}\right) + Z$$
  where:
  - $r$: real FP32 value.
  - $S$: scale factor (positive float).
  - $Z$: zero-point offset (integer).
- **Quantization Range Mapping**:
  $$S = \frac{r_{\max} - r_{\min}}{q_{\max} - q_{\min}}$$
  $$Z = \text{round}\left(\frac{-r_{\min}}{S}\right) + q_{\min}$$
---


In [ ]:
import numpy as np

class ModelCompression:
    """Implements weight pruning and quantization algorithms."""
    @staticmethod
    def prune_weights(weights, percentile=50):
        """Zeroes out weights below the specified magnitude percentile."""
        threshold = np.percentile(np.abs(weights), percentile)
        pruned = np.copy(weights)
        pruned[np.abs(pruned) < threshold] = 0.0
        sparsity = np.mean(pruned == 0.0)
        return pruned, sparsity

    @staticmethod
    def quantize_fp32_to_int8(r):
        """Quantizes an array from FP32 to signed INT8 range [-128, 127]."""
        r_min, r_max = np.min(r), np.max(r)
        q_min, q_max = -128.0, 127.0
        
        # Calculate Scale S
        S = (r_max - r_min) / (q_max - q_min)
        S = max(S, 1e-8)  # Prevent division by zero
        
        # Calculate Zero-point Z
        Z = np.round(( -r_min ) / S) + q_min
        Z = np.clip(Z, q_min, q_max)
        
        # Quantize and clip
        q = np.round(r / S + Z)
        q = np.clip(q, q_min, q_max).astype(np.int8)
        
        return q, S, int(Z)

    @staticmethod
    def dequantize_int8_to_fp32(q, S, Z):
        """Restores FP32 values from quantized representation."""
        return S * (q.astype(np.float32) - Z)



In [ ]:
print("--- Model Pruning Demo ---")
np.random.seed(42)
weights = np.random.randn(4, 4) * 0.5
print("Original Weights:")
print(weights.round(4))

pruned_w, sparsity = ModelCompression.prune_weights(weights, percentile=50)
print(f"\nPruned Weights (Sparsity: {sparsity:.1%}):")
print(pruned_w.round(4))



In [ ]:
print("\n--- Model Quantization Demo ---")
q_w, S, Z = ModelCompression.quantize_fp32_to_int8(weights)
restored_w = ModelCompression.dequantize_int8_to_fp32(q_w, S, Z)

print("Quantized INT8 representation:")
print(q_w)
print(f"Scale: {S:.6f} | Zero-Point: {Z}")

print("\nRestored FP32 representation (Dequantized):")
print(restored_w.round(4))
print(f"Mean absolute reconstruction error: {np.mean(np.abs(weights - restored_w)):.6f}")
